## Reproducibility & AI-assistance disclosure

Part of **`dislocation-speech-translation-fr-en`**, companion to the M2 thesis *Dislocation under Translation: A Corpus Study of Whisper and XLS-R on Spontaneous Spoken French* (Zoé de Vries, Université Paris Cité, 2026; supervised by N. Ballier).

**AI-assistance disclosure.** The executable code is the author's. The explanatory **comments and markdown** were drafted with the help of a large language model (LLM) and reviewed by the author; the LLM changed no executable logic.

**Pipeline order:** `01_whisper` → `02_xlsr` → `03_align` → `04_score`.


# XLS-R — Transcription FR (CTC) + Traduction directe FR→EN, sur grille Whisper

| Étape | Modèle | Tâche |
|-------|--------|-------|
| 1 | `jonatasgrosman/wav2vec2-large-xlsr-53-french` | ASR — transcription FR |
| 2 | `facebook/wav2vec2-xls-r-1b-21-to-en` | ST — traduction directe audio → EN (**1B**, pas 300M) |

Modèles chargés/libérés séquentiellement (limite VRAM T4 16 GB).

**Entrée :** l'enregistrement source complet (même URL que Whisper) **+** le manifest `whisper_grid_manifest.csv` produit par le notebook Whisper et déposé dans le Drive. XLS-R ne fait **pas** de découpage en blocs de 30 s : il applique les bornes canoniques définies par Whisper. Une tranche n'est sous-découpée (sans chevauchement) que si elle dépasse `MAX_SLICE_S` — c'est le confond seg_37 généralisé, désormais uniforme entre modèles et signalé par la colonne `subchunked`.

**Sortie :** `translations_xlsr_aligned.csv` :
`segment_id, t_start, t_end, dur_s, whisper_fr, whisper_en, xlsr_fr, xlsr_en, subchunked`.


In [ ]:
# Mount Google Drive (Colab); force-remount if a stale mount is present.

# ── 0. Montage Google Drive ────────────────────────────────────────────────────
import os
from google.colab import drive
mp = "/content/drive"
if os.path.exists(mp) and os.path.isdir(mp) and os.listdir(mp):
    try:
        drive.flush_and_unmount()
    except ValueError:
        pass
    if os.path.exists(mp) and os.listdir(mp):
        import shutil; shutil.rmtree(mp)
drive.mount(mp, force_remount=True)


In [ ]:
# Config. Reads the Whisper manifest, writes outputs to Drive. CKPT_PATH = checkpoint/resume across
# Colab disconnects; FLUSH_EVERY=25. MAX_SLICE_S=28 s triggers non-overlapping sub-chunking (never fires here).

# ── 1. CONFIG ──────────────────────────────────────────────────────────────────
from pathlib import Path

# Source IDENTIQUE à Whisper (même URL) → audio strictement reproductible.
SOURCE_URL = "http://cfpp2000.univ-paris3.fr/data/public/11eme/Anita_MUSSO_F_46_11e/Anita_MUSSO_F_46_11e.wav"
RAW_PATH   = Path("data/Anita_MUSSO_F_46_11e_raw.wav")

# Manifest produit par le notebook Whisper, déposé dans le Drive (À ADAPTER) :
MANIFEST_PATH = Path("/content/drive/MyDrive/ENS Saclay 5A (M2R)/whisper_grid_manifest.csv")

OUT_DIR = Path("/content/drive/MyDrive/ENS Saclay 5A (M2R)/xlsr_final_final")

SR          = 16_000   # obligatoire pour les deux modèles
MAX_SLICE_S = 28.0     # au-delà : sous-découpage SANS chevauchement (sécurité XLS-R)

OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH   = OUT_DIR / "_ckpt_xlsr.csv"   # reprise après coupure Colab
FLUSH_EVERY = 25                            # sauvegarde Drive toutes les N itérations
print("Checkpoint :", CKPT_PATH, "| existe :", CKPT_PATH.exists())
print("Manifest présent :", MANIFEST_PATH.exists())
print("Output dir       :", OUT_DIR)


In [ ]:
# Report GPU model and available RAM (thesis runs: a single Colab T4).

# ── 2. GPU / RAM ───────────────────────────────────────────────────────────────
import torch, psutil
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU : {p.name} — {p.total_memory/1024**3:.1f} GB")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE, "| RAM dispo :", f"{psutil.virtual_memory().available/1024**3:.1f} GB")


In [ ]:
# Pin dependencies. transformers==4.44.2 is the version used for the reported outputs.

# ── 3. Dépendances ─────────────────────────────────────────────────────────────
!pip install -q transformers==4.44.2 sentencepiece librosa soundfile tqdm accelerate


In [ ]:
# Helpers: no_overlap_subchunks (safety for >28 s slices), clean (whitespace), free_gpu (release VRAM).

# ── 4. Imports + utilitaires ───────────────────────────────────────────────────
import pandas as pd, numpy as np, librosa, gc
from tqdm import tqdm
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor

def no_overlap_subchunks(y, max_s=MAX_SLICE_S, sr=SR):
    """Sous-découpe SANS chevauchement, uniquement si la tranche dépasse max_s."""
    win = int(max_s * sr)
    if len(y) <= win:
        return [y]
    n = int(np.ceil(len(y) / win))
    return [y[i*win:(i+1)*win] for i in range(n)]

def clean(s):
    return " ".join(s.strip().split())

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
# Download the WAV (same URL as notebook 01); load the manifest OR resume from the checkpoint;
# ensure the xlsr_fr/xlsr_en columns exist before processing.

# ── 5. Source + manifest + audio en mémoire ───────────────────────────────────
import requests
if not RAW_PATH.exists():
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Téléchargement source…")
    with requests.get(SOURCE_URL, stream=True, timeout=180) as r:
        r.raise_for_status()
        with open(RAW_PATH, "wb") as f:
            for c in r.iter_content(chunk_size=1024*256):
                f.write(c)
    print("OK :", RAW_PATH)

if CKPT_PATH.exists():
    df = pd.read_csv(CKPT_PATH)
    print(f"REPRISE depuis le checkpoint : {len(df)} lignes.")
else:
    df = pd.read_csv(MANIFEST_PATH)
    print(f"{len(df)} segments canoniques chargés depuis le manifest Whisper.")
assert {"segment_id","t_start","t_end"}.issubset(df.columns), "Manifest/checkpoint invalide"
for _col in ("xlsr_fr","xlsr_en"):
    if _col not in df.columns: df[_col] = ""
    df[_col] = df[_col].fillna("").astype(str)
def _todo(col):
    return [i for i in df.index if not str(df.at[i, col]).strip()]

audio_full, _ = librosa.load(RAW_PATH, sr=SR, mono=True)
print(f"Audio : {len(audio_full)/SR:.1f} s")
n_sub = int((df['t_end'] - df['t_start'] > MAX_SLICE_S).sum())
print(f"Segments > {MAX_SLICE_S}s (seront sous-découpés, sans chevauchement) : {n_sub}")


---
## Étape A — Transcription FR (XLSR-53, CTC)

In [ ]:
# ASR model: jonatasgrosman/wav2vec2-large-xlsr-53-french (CTC, greedy argmax, no LM).

# ── 6. Modèle ASR ──────────────────────────────────────────────────────────────
ASR_ID        = "jonatasgrosman/wav2vec2-large-xlsr-53-french"
asr_processor = Wav2Vec2Processor.from_pretrained(ASR_ID)
asr_model     = Wav2Vec2ForCTC.from_pretrained(ASR_ID).to(DEVICE).eval()
print("ASR chargé sur", DEVICE)


In [ ]:
# ASR pass: French transcription per canonical segment; slices < 0.2 s left empty; checkpointed. -> xlsr_fr.

# ── 7. Transcription FR par segment canonique ──────────────────────────────────
@torch.inference_mode()
def asr_fr(y):
    parts = []
    for sub in no_overlap_subchunks(y):
        inp = asr_processor(sub, sampling_rate=SR, return_tensors="pt")
        iv  = inp.input_values.to(DEVICE)
        am  = inp.get("attention_mask")
        am  = am.to(DEVICE) if am is not None else None
        logits = asr_model(iv, attention_mask=am).logits
        ids    = torch.argmax(logits, dim=-1)
        parts.append(asr_processor.decode(ids[0], skip_special_tokens=True))
    return clean(" ".join(parts))

todo = _todo("xlsr_fr")
print(f"ASR : {len(df)-len(todo)} déjà fait(s), {len(todo)} à traiter")
since = 0
for i in tqdm(todo, desc="ASR FR / segment"):
    row = df.loc[i]
    a = int(row["t_start"] * SR); b = int(row["t_end"] * SR)
    clip = audio_full[a:b]
    if len(clip) < int(0.2 * SR):
        df.at[i, "xlsr_fr"] = ""
    else:
        try:
            df.at[i, "xlsr_fr"] = asr_fr(clip)
        except Exception as e:
            print("  ✗", row["segment_id"], e); df.at[i, "xlsr_fr"] = ""
    since += 1
    if since >= FLUSH_EVERY:
        df.to_csv(CKPT_PATH, index=False, encoding="utf-8"); since = 0
df.to_csv(CKPT_PATH, index=False, encoding="utf-8")
print("ASR terminé — checkpoint à jour :", CKPT_PATH)


In [ ]:
# Release the ASR model from VRAM before loading the larger ST model.

# ── 8. Libération VRAM ────────────────────────────────────────────────────────
del asr_model, asr_processor
free_gpu(); print("VRAM libérée.")


---
## Étape B — Traduction directe FR→EN (XLS-R 1B)

In [ ]:
# ST model: facebook/wav2vec2-xls-r-1b-21-to-en (XLS-R encoder + mBART-50 decoder).

# ── 9. Modèle ST ───────────────────────────────────────────────────────────────
from transformers import (Wav2Vec2FeatureExtractor, MBart50TokenizerFast,
                           SpeechEncoderDecoderModel)
ST_ID = "facebook/wav2vec2-xls-r-1b-21-to-en"
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(ST_ID)
tokenizer         = MBart50TokenizerFast.from_pretrained(ST_ID)
st_model          = SpeechEncoderDecoderModel.from_pretrained(ST_ID).to(DEVICE).eval()
print("ST chargé sur", DEVICE)


In [ ]:
# ST pass: direct FR->EN per segment; beam search (num_beams=4) with no_repeat_ngram_size=3 and
# repetition_penalty=1.3 to curb repetition loops; same guards/checkpointing. -> xlsr_en.

# ── 10. Traduction directe par segment canonique ──────────────────────────────
@torch.inference_mode()
def translate_fr_en(y):
    parts = []
    for sub in no_overlap_subchunks(y):
        inp = feature_extractor(sub, sampling_rate=SR, return_tensors="pt")
        iv  = inp.input_values.to(DEVICE)
        am  = inp.get("attention_mask")
        am  = am.to(DEVICE) if am is not None else None
        gen = st_model.generate(iv, attention_mask=am, num_beams=4,
                                max_length=512, no_repeat_ngram_size=3,
                                repetition_penalty=1.3, early_stopping=True)
        parts.append(tokenizer.batch_decode(gen, skip_special_tokens=True)[0].strip())
    return clean(" ".join(parts))

todo = _todo("xlsr_en")
print(f"ST : {len(df)-len(todo)} déjà fait(s), {len(todo)} à traiter")
since = 0
for i in tqdm(todo, desc="ST FR→EN / segment"):
    row = df.loc[i]
    a = int(row["t_start"] * SR); b = int(row["t_end"] * SR)
    clip = audio_full[a:b]
    if len(clip) < int(0.2 * SR):
        df.at[i, "xlsr_en"] = ""
    else:
        try:
            df.at[i, "xlsr_en"] = translate_fr_en(clip)
        except Exception as e:
            print("  ✗", row["segment_id"], e); df.at[i, "xlsr_en"] = ""
    since += 1
    if since >= FLUSH_EVERY:
        df.to_csv(CKPT_PATH, index=False, encoding="utf-8"); since = 0
df.to_csv(CKPT_PATH, index=False, encoding="utf-8")
print("ST terminé — checkpoint à jour :", CKPT_PATH)


In [ ]:
# Release the ST model from VRAM.

# ── 11. Libération VRAM ───────────────────────────────────────────────────────
del st_model, feature_extractor, tokenizer
free_gpu(); print("VRAM libérée.")


---
## Étape C — Export aligné (4 sorties sur la même grille)

In [ ]:
# Merge and export the shared CSV segment_translations.csv consumed by notebook 03.

# ── 12. Fusion et sauvegarde ──────────────────────────────────────────────────
if "df" not in globals():
    import pandas as pd
    df = pd.read_csv(CKPT_PATH)
cols = ["segment_id","t_start","t_end","dur_s",
        "whisper_fr","whisper_en_per_seg","whisper_en_cont","xlsr_fr","xlsr_en"]
cols = [c for c in cols if c in df.columns]
out = OUT_DIR / "segment_translations.csv"
df[cols].to_csv(out, index=False, encoding="utf-8")
print("CSV final :", out, f"({len(df)} lignes)")
df[cols].head(10)
